# ☕ Prática 1: Engenharia de Dados
### Workshop Dados na Prática · Nerd Data Lab + Programa AI

O dono do **Café Tambaú** quer saber por que as vendas caíram. Mas antes de qualquer análise, alguém precisa garantir que os dados estão **corretos e organizados**. Esse é o trabalho da Engenharia de Dados.

Vamos seguir a **arquitetura Medallion**, usada por muitas empresas:

| Camada | O que é |
|---|---|
| 🥉 **Bronze** | Os dados como chegaram do sistema, sem mexer em nada |
| 🥈 **Prata** | Os dados limpos e padronizados |
| 🥇 **Ouro** | Os dados prontos para o negócio, com as colunas que a análise precisa |

**Como usar este notebook:** clique em uma célula e aperte **Shift + Enter** para rodar. Nas células marcadas com ✏️, troque os `___` pelo código certo.

In [ ]:
# ⚙️ Configuração: rode esta célula primeiro (Shift + Enter)
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

# De onde vêm os dados:
# 1) Se você subiu a pasta cafe_tambau para o Colab (ícone de pasta à esquerda), ela é usada.
# 2) Se não, os arquivos são baixados deste endereço.
URL_BASE = "https://raw.githubusercontent.com/wuldson-franco/workshop_dados_pratica/main/"
PASTA_LOCAL = "./"

def ler(arquivo, **opcoes):
    caminho = PASTA_LOCAL + arquivo
    if os.path.exists(caminho):
        return pd.read_csv(caminho, **opcoes)
    return pd.read_csv(URL_BASE + arquivo, **opcoes)

print("Tudo pronto! ✅")

## 🥉 1. Camada Bronze: carregar os dados brutos

Vamos ler tudo **como texto** (`dtype=str`), do jeito que veio do sistema, para enxergar os problemas.

In [ ]:
vendas = ler("dados_brutos/vendas.csv", dtype=str, keep_default_na=False)
produtos = ler("dados_brutos/produtos.csv", dtype=str, keep_default_na=False)

print("Linhas e colunas de vendas:", vendas.shape)
vendas.head(10)

In [ ]:
produtos

## 🔍 2. Diagnóstico: o que está errado?

Antes de limpar, um bom engenheiro **investiga**. Vamos olhar coluna por coluna.

**Canal de venda:** quantas formas diferentes de escrever "Balcão" e "iFood" existem?

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
vendas["___"].value_counts()

**Forma de pagamento:**

In [ ]:
vendas["forma_pagamento"].value_counts()

**Datas e preços:** olhe uma amostra. Repare nos formatos diferentes.

In [ ]:
vendas[["data_hora", "preco_unitario", "quantidade", "id_cliente"]].sample(15, random_state=1)

**Linhas duplicadas:** o sistema às vezes registra a mesma venda duas vezes.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
vendas.___().sum()

**Linhas de teste:** alguém testou o sistema e esqueceu de apagar.

In [ ]:
vendas[vendas["id_venda"].str.startswith("TESTE")]

📝 **Resumo dos problemas encontrados:**
- Datas em 3 formatos diferentes
- Preço com ponto, vírgula e "R$"
- Canal e forma de pagamento escritos de vários jeitos
- Código de produto em maiúsculo e minúsculo
- Cliente não identificado como vazio, "NA" ou "sem cadastro"
- Quantidades vazias ou impossíveis
- Linhas duplicadas e linhas de teste

Se a gente analisasse esses dados assim, **as conclusões estariam erradas**. Vamos limpar.

## 🥈 3. Camada Prata: limpar e padronizar

### 3.1 Remover duplicatas e linhas de teste

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
prata = vendas.drop_duplicates()
prata = prata[~prata["id_venda"].str.startswith("___")]

print("Linhas antes:", len(vendas), "| depois:", len(prata))

### 3.2 Padronizar as datas

Essa função tenta os três formatos que encontramos. Só rode a célula.

In [ ]:
def converter_data(coluna):
    formatos = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%d-%m-%Y %Hh%M"]
    resultado = pd.Series(pd.NaT, index=coluna.index)
    for formato in formatos:
        convertido = pd.to_datetime(coluna, format=formato, errors="coerce")
        resultado = resultado.fillna(convertido)
    return resultado

prata["data_hora"] = converter_data(prata["data_hora"])
print("Datas que não foi possível converter:", prata["data_hora"].isna().sum())
prata["data_hora"].head()

### 3.3 Transformar o preço em número

Tiramos o "R$", trocamos a vírgula por ponto e convertemos para número.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
def limpar_preco(texto):
    texto = texto.replace("R$", "").strip()
    texto = texto.replace(",", "___")
    return float(texto)

prata["preco_unitario"] = prata["preco_unitario"].apply(limpar_preco)
prata["preco_unitario"].describe()

### 3.4 Padronizar canal e forma de pagamento

Um dicionário diz qual é o jeito certo de escrever cada valor. Complete os que faltam para o iFood.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
mapa_canal = {
    "Balcão": "Balcão", "balcao": "Balcão", "BALCAO": "Balcão", "Balcao": "Balcão",
    "iFood": "iFood", "ifood": "iFood", "___": "iFood", "___": "iFood",
}
prata["canal"] = prata["canal"].map(mapa_canal)
prata["canal"].value_counts(dropna=False)

In [ ]:
mapa_pagamento = {
    "PIX": "Pix", "credito": "Cartão de crédito", "debito": "Cartão de débito",
    "dinheiro": "Dinheiro", "ifood online": "iFood (online)",
}
prata["forma_pagamento"] = prata["forma_pagamento"].replace(mapa_pagamento)
prata["forma_pagamento"].value_counts()

### 3.5 Código do produto em maiúsculo

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
prata["id_produto"] = prata["id_produto"].str.___()
prata["id_produto"].unique()

### 3.6 Cliente não identificado vira vazio de verdade

In [ ]:
prata["id_cliente"] = prata["id_cliente"].replace(["", "NA", "sem cadastro"], None)
print("Vendas sem cliente identificado:", prata["id_cliente"].isna().sum())

### 3.7 Quantidades válidas

Convertemos para número e ficamos só com quantidades entre 1 e 10. Zero, negativo ou 100 cafés num item é erro de digitação.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
prata["quantidade"] = pd.to_numeric(prata["quantidade"], errors="coerce")
invalidas = ~prata["quantidade"].between(1, ___)
print("Linhas com quantidade inválida:", invalidas.sum())

prata = prata[~invalidas]
prata["quantidade"] = prata["quantidade"].astype(int)

In [ ]:
prata["tempo_entrega_min"] = pd.to_numeric(prata["tempo_entrega_min"], errors="coerce")
prata.info()

## 🥇 4. Camada Ouro: dados prontos para o negócio

Agora juntamos as vendas com os produtos (nome, categoria e custo) e criamos as colunas que a análise vai usar: **valor total, lucro, mês, trimestre e período do dia**.

In [ ]:
produtos["custo_unitario"] = produtos["custo_unitario"].str.replace(",", ".").astype(float)
produtos_ouro = produtos[["id_produto", "nome_produto", "categoria", "custo_unitario"]]

ouro = prata.merge(produtos_ouro, on="id_produto", how="left")
ouro.head()

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
ouro["valor_total"] = ouro["quantidade"] * ouro["___"]
ouro["custo_total"] = ouro["quantidade"] * ouro["custo_unitario"]
ouro["lucro"] = ouro["valor_total"] - ouro["___"]

In [ ]:
ouro["data"] = ouro["data_hora"].dt.date
ouro["hora"] = ouro["data_hora"].dt.hour
ouro["mes"] = ouro["data_hora"].dt.month
ouro["trimestre"] = "T" + ouro["data_hora"].dt.quarter.astype(str)

def periodo_do_dia(hora):
    if hora <= 10:
        return "Manhã"
    elif hora <= 14:
        return "Almoço"
    elif hora <= 17:
        return "Tarde"
    return "Noite"

ouro["periodo"] = ouro["hora"].apply(periodo_do_dia)
ouro = ouro.sort_values("data_hora")
ouro.head()

## ✅ 5. Validação: será que limpamos direito?

Comparamos o nosso resultado com a versão tratada oficial.

In [ ]:
oficial = ler("dados_tratados/vendas_limpas.csv")

print(f"Faturamento com dados BRUTOS (sem limpar): impossível calcular, o preço é texto!")
print(f"Nosso faturamento:   R$ {ouro['valor_total'].sum():,.2f}")
print(f"Faturamento oficial: R$ {oficial['valor_total'].sum():,.2f}")
print(f"Nossas linhas: {len(ouro)} | oficial: {len(oficial)}")

A diferença pequena vem das linhas com quantidade inválida que removemos: no mundo real, a gente perguntaria ao time da cafeteria qual era o valor certo.

Uma primeira espiada no faturamento por mês, só para conferir:

In [ ]:
ouro.groupby("mes")["valor_total"].sum().plot(kind="bar", title="Faturamento por mês (R$)", color="#1FD3F0")
plt.show()

## 💾 6. Salvar a camada Ouro

In [ ]:
ouro.to_csv("vendas_ouro.csv", index=False)
print("Arquivo vendas_ouro.csv salvo! Ele aparece no ícone de pasta à esquerda. 🎉")

## ⭐ Desafio bônus: limpar os clientes

O arquivo `clientes.csv` também tem problemas: bairros escritos de vários jeitos, clientes duplicados, idades impossíveis e fidelidade como "S", "N", "sim", "nao". Tente limpar!

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
clientes = ler("dados_brutos/clientes.csv", dtype=str, keep_default_na=False)
print(clientes["bairro"].value_counts())

# Dica: .str.strip() tira espaços, .str.title() deixa a primeira letra maiúscula
clientes["bairro"] = clientes["bairro"].str.___().str.title()
correcao_acentos = {"Tambau": "Tambaú", "Manaira": "Manaíra", "Bancarios": "Bancários"}
clientes["bairro"] = clientes["bairro"].replace(correcao_acentos)

clientes = clientes.drop_duplicates()
clientes["idade"] = pd.to_numeric(clientes["idade"])
clientes.loc[~clientes["idade"].between(___, 100), "idade"] = None

clientes["programa_fidelidade"] = clientes["programa_fidelidade"].replace({"sim": "Sim", "S": "Sim", "nao": "Não", "N": "Não"})
clientes["bairro"].value_counts()

## 🎯 O que a Engenharia de Dados entregou

Transformamos dados bagunçados em uma tabela confiável. **Sem essa etapa, nenhuma análise, modelo ou IA funciona direito.** É por isso que se diz que a maior parte do tempo em um projeto de dados é gasta preparando os dados.

📋 **Não esqueça:** tire um print do resultado da validação e envie no formulário para ganhar seu cupom do sorteio! 🎁